In [1]:
# Cell 1: Imports and Configuration

import pandas as pd
import numpy as np
import psycopg2
from psycopg2 import extras
from dotenv import load_dotenv
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from datetime import datetime, timedelta

# Load environment variables for database connection
# Ensure this path is correct for your .env file
load_dotenv(dotenv_path='/Users/bhanutejamalineni/phis_project/backend/.env') 

DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")

print("Evaluation Notebook: Imports and Configuration Loaded.")

Evaluation Notebook: Imports and Configuration Loaded.


In [2]:
# Cell 2: Database Connection and Data Fetching

HEALTH_DATA_COLUMNS = [
    'id', 'timestamp', 'device_name', 'brand', 'model', 'heart_rate', 'steps',
    'calories', 'activity_level', 'sleep_duration', 'oxygen_saturation',
    'body_temperature', 'blood_pressure_systolic', 'blood_pressure_diastolic'
]

INSIGHTS_COLUMNS = [
    'insight_id', 'timestamp', 'device_name', 'metric_involved', 'anomaly_type',
    'insight_text', 'severity', 'is_read', 'created_at'
]

def get_db_connection():
    """Establishes and returns a database connection."""
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    return conn

def fetch_health_data():
    """Fetches all raw health data from the database."""
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute(f"SELECT {', '.join(HEALTH_DATA_COLUMNS)} FROM health_data ORDER BY timestamp ASC;")
        data = cur.fetchall()
        cur.close()
        conn.close()
        df = pd.DataFrame(data, columns=HEALTH_DATA_COLUMNS)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp').sort_index()
        # Ensure numeric types
        numeric_cols = [col for col in HEALTH_DATA_COLUMNS if col not in ['id', 'timestamp', 'device_name', 'brand', 'model', 'activity_level']]
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        return df
    except Exception as e:
        print(f"Error fetching health data for evaluation: {e}")
        return pd.DataFrame(columns=HEALTH_DATA_COLUMNS).set_index('timestamp')

def fetch_insights_data():
    """Fetches all insights from the database."""
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute(f"SELECT {', '.join(INSIGHTS_COLUMNS)} FROM insights ORDER BY timestamp ASC;")
        data = cur.fetchall()
        cur.close()
        conn.close()
        df = pd.DataFrame(data, columns=INSIGHTS_COLUMNS)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['created_at'] = pd.to_datetime(df['created_at'])
        df = df.set_index('timestamp').sort_index()
        return df
    except Exception as e:
        print(f"Error fetching insights data for evaluation: {e}")
        return pd.DataFrame(columns=INSIGHTS_COLUMNS).set_index('timestamp')

# Fetch data for evaluation
health_df = fetch_health_data()
insights_df = fetch_insights_data()

print(f"\nFetched {len(health_df)} health data entries.")
print(f"Fetched {len(insights_df)} insights.")

if not health_df.empty:
    print("\nHealth Data Head (for evaluation):")
    print(health_df.head())
if not insights_df.empty:
    print("\nInsights Data Head (for evaluation):")
    print(insights_df.head())


Fetched 410380 health data entries.
Fetched 3038 insights.

Health Data Head (for evaluation):
                     id          device_name    brand          model  \
timestamp                                                              
2025-11-16 13:02:14   1  Apple Watch Ultra 2    Apple  Watch Ultra 2   
2025-11-16 13:02:19   2        Amazfit GTR 4  Amazfit          GTR 4   
2025-11-16 13:02:24   3            WHOOP 4.0    WHOOP            4.0   
2025-11-16 13:02:29   4            WHOOP 4.0    WHOOP            4.0   
2025-11-16 13:02:34   5  Apple Watch Ultra 2    Apple  Watch Ultra 2   

                     heart_rate  steps  calories activity_level  \
timestamp                                                         
2025-11-16 13:02:14         163    677        65        Walking   
2025-11-16 13:02:19         154    415       272        Running   
2025-11-16 13:02:24         135    345       242        Resting   
2025-11-16 13:02:29         110    938       123        Cycling 

In [3]:
# Cell 3: System Functionality Test - Data Flow

print("\n--- Testing End-to-End Data Flow ---")

# Check if data is being generated
if health_df.empty:
    print("WARNING: No health data found. Ensure simulator is running.")
else:
    print(f"Health data present for {health_df['device_name'].nunique()} unique devices from {health_df.index.min()} to {health_df.index.max()}.")

# Check if insights are being generated
if insights_df.empty:
    print("WARNING: No insights found. Ensure ML pipeline is running after data simulation.")
else:
    print(f"Insights present for {insights_df['device_name'].nunique()} unique devices from {insights_df.index.min()} to {insights_df.index.max()}.")

# Check integration: Do insights correspond to data timestamps?
if not health_df.empty and not insights_df.empty:
    # Example: Check for insights within a recent time window
    latest_health_timestamp = health_df.index.max()
    recent_insights = insights_df[insights_df.index > (latest_health_timestamp - timedelta(minutes=30))]
    
    if not recent_insights.empty:
        print(f"\nFound {len(recent_insights)} insights generated in the last 30 minutes of data collection.")
        print("This indicates that the ML pipeline is processing recent data.")
    else:
        print("\nNo recent insights found matching the latest health data timestamps. This might indicate a delay in ML processing or lack of new anomalies.")
else:
    print("\nCannot verify insight-data correspondence due to missing data/insights.")

print("End-to-End Data Flow Test Complete.")


--- Testing End-to-End Data Flow ---
Health data present for 29 unique devices from 2025-11-16 13:02:14 to 2025-12-10 07:00:29.
Insights present for 29 unique devices from 2025-11-16 14:01:44 to 2025-12-10 06:54:14.

Found 4 insights generated in the last 30 minutes of data collection.
This indicates that the ML pipeline is processing recent data.
End-to-End Data Flow Test Complete.


In [4]:
# Cell 4: Performance Assessment - Anomaly Detection (Qualitative/Sample-based)

print("\n--- Anomaly Detection Performance Assessment (Qualitative) ---")

if insights_df.empty:
    print("No insights to assess. Please run the ML pipeline.")
else:
    print("\nReviewing sample insights and their associated data points:")

    # Get a sample of recent insights, focusing on different anomaly types
    sample_insights = insights_df.sort_values(by='created_at', ascending=False).drop_duplicates(subset=['anomaly_type', 'device_name']).head(5)

    if sample_insights.empty:
        print("No diverse insights found for sampling.")
    else:
        for idx, insight_row in sample_insights.iterrows():
            device = insight_row['device_name']
            insight_ts = insight_row.name # The timestamp index
            anomaly_type = insight_row['anomaly_type']
            insight_text = insight_row['insight_text']

            print(f"\nInsight ID: {insight_row['insight_id']} | Device: {device} | Timestamp: {insight_ts} | Type: {anomaly_type}")
            print(f"  Insight: {insight_text}")

            # Try to fetch the corresponding health data point
            # We'll look for data around the insight's timestamp
            time_window_start = insight_ts - timedelta(minutes=5)
            time_window_end = insight_ts + timedelta(minutes=5)
            relevant_health_data = health_df[(health_df.index >= time_window_start) & (health_df.index <= time_window_end) & (health_df['device_name'] == device)]

            if not relevant_health_data.empty:
                print("  Relevant Health Data (closest point to insight timestamp):")
                print(relevant_health_data.iloc[0].to_string()) # Display the closest data point
                # You could add logic here to compare actual value with baseline mean/std
                # e.g., if anomaly_type is 'high_hr', check if heart_rate is high
                print(f"  Is {insight_row['metric_involved']} looking anomalous around this time? (Manual check required)")
            else:
                print("  No direct health data found corresponding to this insight timestamp.")

    print("\nAnomaly detection assessment requires careful manual review of reported anomalies against raw data trends.")
    print("Consider using the 'Raw Data Explorer' in the dashboard to visually verify these points.")

# --- System Speed/Latency (Conceptual) ---
print("\n--- System Speed & Latency Assessment (Conceptual) ---")
print("Evaluating system speed primarily involves observing the simulator's throughput,")
print("the ML pipeline's processing time, and dashboard responsiveness.")
print("The current setup is client-side for evaluation, so actual latency for the ML part would be measured on the backend.")
print("For this project, we assume that the local setup provides near real-time feedback.")
print("Dashboard responsiveness can be manually observed during interaction.")
print("Consider adding logging with timestamps in Flask and ML scripts to get actual processing times.")


--- Anomaly Detection Performance Assessment (Qualitative) ---

Reviewing sample insights and their associated data points:

Insight ID: 2454 | Device: Huawei Band 9 | Timestamp: 2025-11-29 17:24:19 | Type: high_temp
  Insight: Your body temperature (38.2°C) is elevated. This could indicate a fever or strenuous activity. Monitor yourself.
  Relevant Health Data (closest point to insight timestamp):
id                                 227786
device_name                 Huawei Band 9
brand                              Huawei
model                              Band 9
heart_rate                            173
steps                                 486
calories                              291
activity_level                    Resting
sleep_duration                       5.32
oxygen_saturation                    97.0
body_temperature                     38.2
blood_pressure_systolic               140
blood_pressure_diastolic               82
  Is body_temperature looking anomalous around this

In [5]:
# Cell 5: Usability Feedback (Conceptual)

print("\n--- Usability Feedback Gathering (Conceptual) ---")
print("For a real-world project, usability feedback would be gathered through:")
print("1. User interviews: Asking users to interact with the dashboard and provide qualitative feedback.")
print("2. Surveys: Collecting structured feedback on ease of use, helpfulness of insights, etc.")
print("3. Observational studies: Watching users interact with the system to identify pain points.")
print("\nKey questions for feedback:")
print(" - Is the dashboard easy to navigate?")
print(" - Are the insights clear and understandable?")
print(" - Are the recommendations actionable?")
print(" - Does the system feel responsive?")
print(" - What features would improve your experience?")
print("\nSince this is a prototype, self-reflection and feedback from your supervisor/peers can serve this purpose.")


--- Usability Feedback Gathering (Conceptual) ---
For a real-world project, usability feedback would be gathered through:
1. User interviews: Asking users to interact with the dashboard and provide qualitative feedback.
2. Surveys: Collecting structured feedback on ease of use, helpfulness of insights, etc.
3. Observational studies: Watching users interact with the system to identify pain points.

Key questions for feedback:
 - Is the dashboard easy to navigate?
 - Are the insights clear and understandable?
 - Are the recommendations actionable?
 - Does the system feel responsive?
 - What features would improve your experience?

Since this is a prototype, self-reflection and feedback from your supervisor/peers can serve this purpose.


In [6]:
# Cell 6: Documentation Outline and Potential Enhancements

print("\n--- Documentation Outline ---")
print("A comprehensive project report and documentation would include:")
print("1.  **Project Overview:** Introduction, problem statement, goals, methodology.")
print("2.  **System Architecture:** Detailed description of Data Source, Backend, ML, Visualization, Security layers.")
print("3.  **Technical Design:**")
print("    *   Database Schema: ERD and table definitions.")
print("    *   API Endpoints: For data ingestion and insight retrieval.")
print("    *   ML Workflow: Data preprocessing, baseline, anomaly detection algorithms, insight rules.")
print("    *   Dashboard UI/UX: Key components and interactions.")
print("4.  **Implementation Details:** Code structure, key libraries, development environment setup.")
print("5.  **Evaluation Results:** Summarize findings from functional and performance tests.")
print("6.  **Challenges & Mitigation:** Discuss encountered issues and how they were resolved.")
print("7.  **Future Work & Enhancements:**")
print("    *   **ML Model Refinement:** Exploring more advanced models (e.g., LSTMs for time series).")
print("    *   **Contextual Insights:** Incorporating user-reported activities or health conditions.")
print("    *   **Real-time Processing:** Optimizing the ML pipeline for true streaming data.")
print("    *   **User Management:** More robust user profiles and role-based access control.")
print("    *   **Mobile App Integration:** Developing a companion mobile application.")
print("    *   **Cloud Deployment:** Scaling the system for larger datasets and wider accessibility.")
print("    *   **Feedback Loop:** Implementing mechanisms for users to rate insights.")
print("8.  **Conclusion:** Summary of achievements and future outlook.")

print("\n--- Potential Issues & Enhancement Ideas (as identified during development) ---")
print("1.  **Data Quality:** Missing or noisy data from simulators can impact anomaly detection. Mitigation: More sophisticated imputation, outlier smoothing.")
print("2.  **Anomaly Definition:** Z-score and Isolation Forest are general. Domain-specific anomaly patterns (e.g., sudden drop in HR during sleep) could be hard to catch. Enhancement: Rule-based systems for specific health events.")
print("3.  **Scalability:** Current local setup might struggle with millions of data points/users. Enhancement: Distributed computing (Spark), cloud databases (AWS RDS, Google Cloud SQL).")
print("4.  **Personalization:** Baselines are rolling averages. Better personalization might involve more complex user profiling. Enhancement: User-defined thresholds, feedback-driven model adjustments.")
print("5.  **Feedback Loop:** How do we know if an insight was helpful? Enhancement: 'Was this insight useful?' buttons in dashboard, leading to model retraining.")
print("6.  **Security:** Local JWT/session management is good. For cloud, more robust API security and data encryption at rest/transit. Enhancement: Fine-grained access control, regular security audits.")

print("\nPHIS System Integration & Evaluation (Phase 6) Outline Complete.")
print("You now have a framework for testing, assessing, and documenting your project.")


--- Documentation Outline ---
A comprehensive project report and documentation would include:
1.  **Project Overview:** Introduction, problem statement, goals, methodology.
2.  **System Architecture:** Detailed description of Data Source, Backend, ML, Visualization, Security layers.
3.  **Technical Design:**
    *   Database Schema: ERD and table definitions.
    *   API Endpoints: For data ingestion and insight retrieval.
    *   ML Workflow: Data preprocessing, baseline, anomaly detection algorithms, insight rules.
    *   Dashboard UI/UX: Key components and interactions.
4.  **Implementation Details:** Code structure, key libraries, development environment setup.
5.  **Evaluation Results:** Summarize findings from functional and performance tests.
6.  **Challenges & Mitigation:** Discuss encountered issues and how they were resolved.
7.  **Future Work & Enhancements:**
    *   **ML Model Refinement:** Exploring more advanced models (e.g., LSTMs for time series).
    *   **Contextua